# Generate and Evaluate New Model Feedback

This notebook generates feedback using the trained DPO adapter and scores it using the judge model. It also provides steps to upload the results and the adapter to Google Drive and Google Cloud Storage (GCS).

## 1. Setup Environment

Mount Google Drive, clone the repository, and install necessary dependencies.

In [1]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

DRIVE_BASE_PATH = Path('/content/drive/MyDrive/ai-interview-coach')
DRIVE_BASE_PATH.mkdir(parents=True, exist_ok=True)
print(f'Google Drive results folder ready at: {DRIVE_BASE_PATH}')

Mounted at /content/drive
Google Drive results folder ready at: /content/drive/MyDrive/ai-interview-coach


In [3]:
import os
from pathlib import Path

# Clone the repository into Google Drive for persistence
PROJECT_PATH = DRIVE_BASE_PATH
REPO_URL = 'https://github.com/dcyforjob2020/ai-interview-coach.git'

if PROJECT_PATH.exists():
    %cd {PROJECT_PATH}
    print('\n--- Git Status Before Pull ---\n')
    !pwd
    !git status
    !git fetch origin main
    !git checkout main
    # Remove the untracked file that is causing the merge conflict
    !rm train/preference_pairs.jsonl
    !git pull origin main
    print('\n--- Git Status After Pull ---\n')
    !git status
else:
    !git clone {REPO_URL} {PROJECT_PATH}
    %cd {PROJECT_PATH}

print('Project path:', PROJECT_PATH)

/content/drive/MyDrive/ai-interview-coach

--- Git Status Before Pull ---

/content/drive/MyDrive/ai-interview-coach
On branch main
Your branch is behind 'origin/main' by 16 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	train/preference_pairs.jsonl

nothing added to commit but untracked files present (use "git add" to track)
From https://github.com/dcyforjob2020/ai-interview-coach
 * branch            main       -> FETCH_HEAD
Already on 'main'
Your branch is behind 'origin/main' by 16 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/dcyforjob2020/ai-interview-coach
 * branch            main       -> FETCH_HEAD
Updating 322654d..b793734
Updating files: 100% (23/23), done.
Fast-forward
 README.md                                          |     48 +-
 .../baseline}/generate_baseline_feedback.ipynb     |      0
 ..

In [7]:
!pip install -r requirements.txt

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-j9omc4il
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-j9omc4il
  Resolved https://github.com/huggingface/transformers.git to commit 032db9c8d6c3c3cb89e71cc414bfb5a469b1a6da
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.9 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.10.0.dev0-py3-none-any.whl size=12161716 sha256=12ecb4d3e219d7b9dfeb5f0140c723fcae7242ee8cb6549bcfff099d5f23ebb1
  Stored in directory: /tmp/pip-ephem-wheel-

## 2. Download DPO Adapter

Download the fine-tuned DPO adapter from Hugging Face.

In [8]:
from huggingface_hub import snapshot_download
from pathlib import Path

DPO_ADAPTER_REPO = 'Hank122222222/ai-interview-coach-dpo-adapter'
local_dpo_adapter = Path('train/model/dpo_adapter')
local_dpo_adapter.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id=DPO_ADAPTER_REPO,
    repo_type='model',
    local_dir=str(local_dpo_adapter),
    local_dir_use_symlinks=False,
)

print('Downloaded DPO adapter to:', local_dpo_adapter)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Downloaded DPO adapter to: train/model/dpo_adapter


## 3. Generate Feedback with New Model

Run the generation script on the test set.

In [4]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


After running the above `pip install` command, please restart your Colab runtime (Runtime -> Restart runtime) to ensure the new `torchao` version is loaded correctly. Then, run all cells below.

In [5]:
!pip install --upgrade --force-reinstall torchao>=0.16.0

In [6]:
import torchao
print(f"torchao version: {torchao.__version__}")


torchao version: 0.17.0


In [9]:
!python newModel/generate_new_model_feedback.py \
    --input data/test.jsonl \
    --output newModel/new_model_outputs.jsonl \
    --adapter train/model/dpo_adapter \
    --feedback-field new_model_feedback

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Loading base model: Qwen/Qwen2.5-3B-Instruct
Loading adapter: train/model/dpo_adapter
config.json: 100% 661/661 [00:00<00:00, 3.72MB/s]
model.safetensors.index.json: 100% 35.6k/35.6k [00:00<00:00, 94.5MB/s]
Fetching 2 files: 100% 2/2 [00:16<00:00,  8.36s/it]
Download complete: 100% 6.17G/6.17G [00:16<00:00, 368MB/s]
Loading weights: 100% 434/434 [00:01<00:00, 277.84it/s]
generation_config.json: 100% 242/242 [00:00<00:00, 1.57MB/s]
[23/200] Generating new model feedback for test_0023
[24/200] Generating new model feedback for test_0024
[25/200] Generating new model feedback for test_0

## 4. Score New Model Feedback

Use the judge model (Qwen3.5-9B) to score the generated feedback.

In [10]:
!python eval/score_feedback.py \
    --input newModel/new_model_outputs.jsonl \
    --output newModel/new_model_scores.jsonl \
    --feedback-field new_model_feedback \
    --model Qwen/Qwen3.5-9B \
    --quantize

Loading judge model: Qwen/Qwen3.5-9B (quantize=True)
config.json: 100% 3.13k/3.13k [00:00<00:00, 9.81MB/s]
tokenizer_config.json: 100% 16.7k/16.7k [00:00<00:00, 45.2MB/s]
vocab.json: 100% 6.72M/6.72M [00:00<00:00, 127MB/s]
merges.txt: 100% 3.35M/3.35M [00:00<00:00, 100MB/s]
tokenizer.json: 100% 12.8M/12.8M [00:01<00:00, 10.5MB/s]
chat_template.jinja: 100% 7.76k/7.76k [00:00<00:00, 22.6MB/s]
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
model.safetensors.index.json: 100% 79.7k/79.7k [00:00<00:00, 144MB/s]
Fetching 4 files: 100% 4/4 [00:52<00:00, 13.20s/it]
Download complete: 100% 19.3G/19.3G [00:52<00:00, 525MB/s]                